# Virtual flow metering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/olaflaitinen/wellcast/blob/main/notebooks/03_virtual_flow_metering.ipynb)

**Author:** Olaf Yunus Laitinen Imanov ([ORCID 0009-0006-5184-0810](https://orcid.org/0009-0006-5184-0810))  
**Affiliation:** Department of Mathematics and Statistics, University of Helsinki  
**License:** Mozilla Public License 2.0 (MPL-2.0), Copyright (c) 2026 Olaf Yunus Laitinen Imanov

Predict bottomhole pressure from surface measurements with machine learning, validate the model honestly and explain what drives it.

In [ ]:
# Install wellcast when it is not available yet (for example on Google Colab).
import importlib.util

from IPython import get_ipython

if importlib.util.find_spec("wellcast") is None:
    get_ipython().run_line_magic(
        "pip",
        'install -q "wellcast[all] @ git+https://github.com/olaflaitinen/wellcast.git"',
    )

## Features

`build_features` adds physics features (GOR, water cut, drawdown and so on) and lag and rolling features. Lag and rolling features only use earlier rows of the same well, so they cannot leak the future. Same-row features computed from the target are removed automatically by the `Predictor`.

In [ ]:
import numpy as np

import wellcast
from wellcast.features import build_features, features_using
from wellcast.ml import Predictor
from wellcast.plotting import predicted_vs_actual, prediction_interval_plot

production = wellcast.make_synthetic_field(n_wells=5, years=4, n_injectors=0).production
data = build_features(production)
print("features derived from bhp on the same row:", features_using("bhp"))
data.shape

## Honest validation

Two questions need different tests: how well does the model extrapolate in time (time-ordered splits on calendar dates), and how well does it transfer to a well it has never seen (leave one well out).

In [ ]:
model = Predictor("bhp")
cv_time = model.cross_validate(data, "timeseries", n_splits=4)
cv_time[cv_time["well"] == "all"].round(3)

In [ ]:
cv_well = model.cross_validate(data, "lowo")
cv_well[cv_well["well"] != "all"].round(3)

## Hold-out period with prediction intervals

Train on the first 80 percent of dates and predict the rest. Two quantile models give an 80 percent interval.

In [ ]:
dates = np.sort(data.index.unique())
cutoff = dates[int(0.8 * len(dates))]
train, test = data[data.index < cutoff], data[data.index >= cutoff]
test = test[test["on_stream_hours"] > 0]
model.fit(train)
interval = model.predict_interval(test)
inside = (test["bhp"] >= interval["lower"]) & (test["bhp"] <= interval["upper"])
print(f"interval coverage on the hold-out period: {inside.mean():.1%}")
model.score(test).round(3)

In [ ]:
predicted_vs_actual(test["bhp"], interval["prediction"], test["well"])

In [ ]:
well = test["well"] == "P-02"
prediction_interval_plot(
    test.loc[well, "bhp"], interval[well], title="P-02 bottomhole pressure"
)

## What drives the prediction

SHAP values split each prediction into feature contributions (exact for tree models).

In [ ]:
model.explain(test, max_samples=300, top=12)

## Saving the model

Predictors are saved with joblib. Joblib files are pickles: only load files you trust.

In [ ]:
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "vfm_bhp.joblib"
model.save(path)
Predictor.load(path)